# P4 — RAG Pipeline Evaluation Grid (Colab)

Runs the 12-cell baseline grid (3 chunking × 2 embedding × 2 retrieval) on 100 arXiv papers.

**Estimated time:** ~15–30 min on T4 GPU vs 2–3 hrs on CPU.

---

## Before Running

1. Set runtime to T4 GPU: `Runtime → Change runtime type → T4 GPU`
2. Upload the `newline/projects` folder to Google Drive at `My Drive/newline/projects/`
   (or adjust `PROJECTS_ROOT` in cell 1 below)
3. Run all cells top to bottom

---

## After a Disconnect (Resume)

Colab free tier disconnects after ~90 min of browser inactivity. **Your progress is safe** — results and indices are saved to Drive.

**To resume:**
1. Re-run cells 1–3 (mount Drive, set paths, pip install)
2. Re-run the eval cell — completed cells are **skipped automatically** (`force=False`)

**What survives a disconnect (on Drive):**
- `experiments/results/*.json` — one file per completed grid cell
- `data/indices/*/` — built FAISS indices, reused on resume

**What rebuilds after an interrupted ingest:**
- If disconnect happens mid-ingest (e.g. 10 min into an 18-min mpnet cell), that cell's `index.faiss` won't exist yet → that one cell rebuilds from scratch. All other completed cells are skipped.

> **Tip:** Keep the Colab tab open and scroll occasionally to prevent inactivity timeout, or upgrade to Colab Pro for 24-hr sessions.

## 1 — Mount Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

# ── Adjust if your Drive layout differs ────────────────────────────────────
PROJECTS_ROOT = Path('/content/drive/MyDrive/newline/projects')
# ───────────────────────────────────────────────────────────────────────────

PROJECT_DIR  = PROJECTS_ROOT / 'rag_pipeline_experimentation'
RAG_COMMON   = PROJECTS_ROOT / 'rag_common'
LLM_UTILS    = PROJECTS_ROOT / 'llm_utils'

for p in [PROJECT_DIR, RAG_COMMON, LLM_UTILS]:
    assert p.exists(), f'Not found: {p}'
    sys.path.insert(0, str(p))

print('Paths OK')
print('Project :', PROJECT_DIR)
print('Papers  :', len(list((PROJECT_DIR / 'data/papers').glob('*.pdf'))), 'PDFs')
print('Qrels   :', (PROJECT_DIR / 'data/qrels_filtered.json').exists())

## 2 — Install dependencies

In [ ]:
import subprocess, sys

# faiss-gpu for T4 speedup; falls back gracefully to CPU ops if GPU unavailable
pkgs = [
    'faiss-gpu',
    'sentence-transformers>=2.2',
    'rank-bm25>=0.2',
    'nltk>=3.8',
    'pymupdf>=1.23',
    'openai>=1.0',
    'instructor>=1.0',
    'rich>=13.0',
    'pydantic>=2.0',
    'pyyaml>=6.0',
    'python-dotenv>=1.0',
    'tiktoken>=0.5',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('Install complete')

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

## 3 — Verify GPU

In [ ]:
import torch
gpu = torch.cuda.is_available()
print('GPU available:', gpu)
if gpu:
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — embedding will be slow (~2–3 hrs). Switch runtime to T4.')

## 4 — Run evaluation grid

In [ ]:
import os
os.chdir(PROJECT_DIR)

# Results and indices saved to Drive so they persist across sessions
RESULTS_DIR = PROJECT_DIR / 'experiments/results'
INDEX_DIR   = PROJECT_DIR / 'data/indices'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print('Results :', RESULTS_DIR)
print('Indices :', INDEX_DIR)
print('Existing results:', len(list(RESULTS_DIR.glob('*.json'))), '/ 12')

In [ ]:
from src.config import build_grid_from_yaml
from src.evaluator import load_qrels, filter_qrels_by_docs, best_config
from src.experiment import run_grid
from src.models import ExperimentResult
from rich.console import Console
from rich.table import Table

console = Console()

pdf_paths = sorted((PROJECT_DIR / 'data/papers').glob('*.pdf'))
qrels_raw = load_qrels(str(PROJECT_DIR / 'data/qrels_filtered.json'))
doc_ids   = {p.stem for p in pdf_paths}
qrels     = filter_qrels_by_docs(qrels_raw, doc_ids)
configs   = build_grid_from_yaml(str(PROJECT_DIR / 'config/experiments/baseline.yaml'))

print(f'PDFs: {len(pdf_paths)}  |  Queries: {len(qrels)}  |  Grid cells: {len(configs)}')

done = [0]
def progress(i, total, exp_id):
    done[0] = i
    print(f'[{i+1}/{total}] {exp_id}')

results = run_grid(
    configs=configs,
    pdf_paths=pdf_paths,
    qrels=qrels,
    result_dir=RESULTS_DIR,
    index_base_dir=INDEX_DIR,
    force=False,   # resume — skip cells already in results/
    progress_cb=progress,
)

print(f'\nDone — {len(results)} cells complete')

## 5 — Results table

In [ ]:
ranked = sorted(results, key=lambda r: r.metrics.get("mrr", 0), reverse=True)

header = f"{'Rank':<5} {'Experiment ID':<50} {'MRR':>7} {'R@5':>7} {'NDCG@5':>8} {'Latency':>9}"
print(header)
print("-" * 90)
for i, r in enumerate(ranked):
    m = r.metrics
    row = (
        f"{i+1:<5} {r.experiment_id:<50}"
        f" {m.get('mrr', 0):>7.4f}"
        f" {m.get('recall@5', 0):>7.4f}"
        f" {m.get('ndcg@5', 0):>8.4f}"
        f" {r.avg_latency_s * 1000:>7.1f}ms"
    )
    print(row)

best = ranked[0]
print(f"\nBest config : {best.experiment_id}")
print(f"MRR={best.metrics.get('mrr', 0):.4f}  Recall@5={best.metrics.get('recall@5', 0):.4f}  NDCG@5={best.metrics.get('ndcg@5', 0):.4f}")

# Gate checks
m = best.metrics
gates = {
    "Recall@5 > 0.80":    m.get("recall@5", 0)    > 0.80,
    "Precision@5 > 0.60": m.get("precision@5", 0) > 0.60,
    "MRR > 0.70":         m.get("mrr", 0)          > 0.70,
    "NDCG@5 > 0.75":      m.get("ndcg@5", 0)       > 0.75,
}
print("\nGate checks (best config):")
for label, passed in gates.items():
    status = "PASS" if passed else "FAIL"
    print(f"  {status}  {label}")

## 6 — (Optional) Run LLM judge on best config

Requires Groq API key. Scores generated answers on Relevance, Accuracy, Completeness, Citation Quality.

In [ ]:
import os
from getpass import getpass

# Set API keys — use Colab Secrets (key icon in sidebar) or enter here
os.environ.setdefault('LLM_API_KEY',       getpass('Groq API key (generation): '))
os.environ.setdefault('LLM_JUDGE_API_KEY', os.environ.get('LLM_API_KEY', ''))
os.environ.setdefault('LLM_BASE_URL',       'https://api.groq.com/openai/v1')
os.environ.setdefault('LLM_JUDGE_BASE_URL', 'https://api.groq.com/openai/v1')
os.environ.setdefault('LLM_MODEL',          'llama-3.1-8b-instant')
os.environ.setdefault('LLM_JUDGE_MODEL',    'llama-3.1-8b-instant')
os.environ.setdefault('LLM_RATE_LIMIT_DELAY', '15.0')

In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, 'scripts/run_full_pipeline.py',
    '--result-dir', str(RESULTS_DIR),
    '--index-dir',  str(INDEX_DIR),
    '--qrels',      str(PROJECT_DIR / 'data/qrels_filtered.json'),
    '--viz-dir',    str(PROJECT_DIR / 'visualizations'),
    '--log',        str(PROJECT_DIR / 'experiments/iteration_log.jsonl'),
    '--n-queries',  '10',
    '--delay',      '15',
])